# Train and segment a tiled area

This notebook shows a tiled `samstars` workflow:

- train one model bundle from a clipped labeled training package
- build prepared tiles from larger segmentation rasters
- segment those prepared tiles with the trained bundle

Use this pattern when the segmentation area should be processed tile by tile.


## Inputs

Training uses the clipped labeled training package:

- `data/training_data_strong/img.tif`
- `data/training_data_strong/chm.tif`
- `data/training_data_strong/den.tif`
- `data/training_data_strong/crowns.gpkg`

Segmentation uses the larger clipped rasters, which are prepared into tiled inputs.


In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "samstars").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from samstars import train, segment
from samstars.data import build_tiles


## Set paths


In [ ]:
data_dir = repo_root / "data"
training_data_dir = data_dir / "training_data_strong"

training_img = training_data_dir / "img.tif"
training_chm = training_data_dir / "chm.tif"
training_den = training_data_dir / "den.tif"
training_crowns = training_data_dir / "crowns.gpkg"

img = data_dir / "img_clipped.tif"
chm = data_dir / "chm_clipped.tif"
den = data_dir / "den_clipped.tif"

sam_base_ckpt = repo_root / "sam/sam_vit_l_0b3195.pth"
work_dir = data_dir / "train_and_segment_tiled_area_run"
model_dir = work_dir / "model"
tiles_dir = work_dir / "tiles"
inference_run_dir = work_dir / "segmentation_run"

print("repo_root:", repo_root)
print("training_data_dir:", training_data_dir)
print("model_dir:", model_dir)
print("tiles_dir:", tiles_dir)
print("inference_run_dir:", inference_run_dir)


## Set tutorial parameters


In [ ]:
rgb_bands = (1, 2, 3)

device = "cpu"
storage = "persistent"

tile_size_m = 500.0
tile_overlap_m = 20.0

chip_size = 512
chip_stride = 384

sam_epochs = 3
stardist_epochs = 10
val_fraction = 0.2

crop_smooth_sigma = 2.0
smooth_sigma = 4.0
post_smooth_sigma = 0.0

thin_seeds = False
thin_dist = 0.8
seed_d_min = 4.0
seed_min_dist_px = 4
seed_gauss_sigma = 1
seed_min_samples = 1
seed_merge_radius = 3.0
seed_min_canopy_height = 2.0

stardist_input_key = "index"
stardist_use_aux = True
stardist_prob_thr = 0.3
stardist_nms_thr = 0.4
stardist_window_size_m = 150.0
stardist_window_overlap_m = 30.0
stardist_dedup_iou_thr = 0.6


## Train model bundle

This keeps training products under `model_dir/_training/` and writes the model bundle files at the top level of `model_dir`.


In [ ]:
model = train(
    img=training_img,
    chm=training_chm,
    den=training_den,
    crowns=training_crowns,
    sam_base_ckpt=sam_base_ckpt,
    out=model_dir,
    storage=storage,
    rgb_bands=rgb_bands,
    device=device,
    chip_size=chip_size,
    chip_stride=chip_stride,
    sam_epochs=sam_epochs,
    stardist_epochs=stardist_epochs,
    val_fraction=val_fraction,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    thin_seeds=thin_seeds,
    thin_dist=thin_dist,
    seed_d_min=seed_d_min,
    seed_min_dist_px=seed_min_dist_px,
    seed_gauss_sigma=seed_gauss_sigma,
    seed_min_samples=seed_min_samples,
    seed_merge_radius=seed_merge_radius,
    seed_min_canopy_height=seed_min_canopy_height,
    stardist_input_key=stardist_input_key,
    stardist_use_aux=stardist_use_aux,
    stardist_window_size_m=stardist_window_size_m,
    stardist_window_overlap_m=stardist_window_overlap_m,
    stardist_dedup_iou_thr=stardist_dedup_iou_thr,
    fast=False,
)


## Inspect bundle and persistent training outputs


In [ ]:
sorted(p.name for p in model.root.iterdir())


In [ ]:
training_proposals_dir = model.root / "_training" / "training_run" / "proposals" / "rasters"
sorted(p.name for p in training_proposals_dir.glob("*.tif"))[:40]


## Build inference tiles


In [ ]:
build_tiles(
    img=img,
    chm=chm,
    den=den,
    rgb_bands=rgb_bands,
    out=tiles_dir,
    tile=tile_size_m,
    overlap=tile_overlap_m,
    thin_seeds=thin_seeds,
    thin_dist=thin_dist,
    seed_d_min=seed_d_min,
    seed_min_dist_px=seed_min_dist_px,
    seed_gauss_sigma=seed_gauss_sigma,
    seed_min_samples=seed_min_samples,
    seed_merge_radius=seed_merge_radius,
    seed_min_canopy_height=seed_min_canopy_height,
    skip_existing=True,
)


## Segment tiled inference area


In [ ]:
result = segment(
    model=model,
    out=inference_run_dir,
    storage=storage,
    tiles_dir=tiles_dir,
    device=device,
    stardist_input_key=stardist_input_key,
    stardist_use_aux=stardist_use_aux,
    stardist_prob_thr=stardist_prob_thr,
    stardist_nms_thr=stardist_nms_thr,
    stardist_window_size_m=stardist_window_size_m,
    stardist_window_overlap_m=stardist_window_overlap_m,
    stardist_dedup_iou_thr=stardist_dedup_iou_thr,
    crop_smooth_sigma=crop_smooth_sigma,
    smooth_sigma=smooth_sigma,
    post_smooth_sigma=post_smooth_sigma,
    fast=False,
)


## Final output


In [ ]:
result.crowns
